# Summarize Text

## Overview

## Quickstart

In [ ]:
# Set env var OPENAI_API_KEY or load from a .env file
import dotenv

dotenv.load_dotenv()

In [2]:
import os

os.environ["LANGCHAIN_TRACING_V2"] = "True"

In [3]:
from langchain.chains.summarize import load_summarize_chain
from langchain_community.document_loaders import WebBaseLoader
from langchain_openai import ChatOpenAI

loader = WebBaseLoader("https://lilianweng.github.io/posts/2023-06-23-agent/")
docs = loader.load()

llm = ChatOpenAI(temperature=0, model_name="gpt-3.5-turbo-1106")
chain = load_summarize_chain(llm, chain_type="stuff")

result = chain.invoke(docs)

print(result["output_text"])

The article discusses the concept of LLM-powered autonomous agents, which use large language models as their core controllers. It covers the components of these agents, including planning, memory, and tool use, as well as case studies and proof-of-concept examples. The challenges and limitations of using natural language interfaces for these agents are also addressed. The article provides citations and references for further reading.


## Option 1. Stuff

In [4]:
from langchain.chains.combine_documents.stuff import StuffDocumentsChain
from langchain.chains.llm import LLMChain
from langchain_core.prompts import PromptTemplate

# Define prompt
prompt_template = """Write a concise summary of the following:
"{text}"
CONCISE SUMMARY:"""
prompt = PromptTemplate.from_template(prompt_template)

# Define LLM chain
llm = ChatOpenAI(temperature=0, model_name="gpt-3.5-turbo-16k")
llm_chain = LLMChain(llm=llm, prompt=prompt)

# Define StuffDocumentsChain
stuff_chain = StuffDocumentsChain(llm_chain=llm_chain, document_variable_name="text")

docs = loader.load()
print(stuff_chain.invoke(docs)["output_text"])

/home/vscode/.local/lib/python3.10/site-packages/langchain_core/_api/deprecation.py:139: LangChainDeprecationWarning: The class `LLMChain` was deprecated in LangChain 0.1.17 and will be removed in 1.0. Use RunnableSequence, e.g., `prompt | llm` instead.
  warn_deprecated(


The article discusses the concept of building autonomous agents powered by large language models (LLMs). It explores the components of such agents, including planning, memory, and tool use. The article provides case studies and proof-of-concept examples of LLM-powered agents in various domains. It also highlights the challenges and limitations of using LLMs in autonomous agents.


### Go deeper

## Option 2. Map-Reduce

In [5]:
from langchain.chains import MapReduceDocumentsChain, ReduceDocumentsChain
from langchain_text_splitters import CharacterTextSplitter

llm = ChatOpenAI(temperature=0)

# Map
map_template = """The following is a set of documents
{docs}
Based on this list of docs, please identify the main themes 
Helpful Answer:"""
map_prompt = PromptTemplate.from_template(map_template)
map_chain = LLMChain(llm=llm, prompt=map_prompt)

In [6]:
from langchain import hub

map_prompt = hub.pull("rlm/map-prompt")
map_chain = LLMChain(llm=llm, prompt=map_prompt)

In [7]:
# Reduce
reduce_template = """The following is set of summaries:
{docs}
Take these and distill it into a final, consolidated summary of the main themes. 
Helpful Answer:"""
reduce_prompt = PromptTemplate.from_template(reduce_template)

In [8]:
# Note we can also get this from the prompt hub, as noted above
reduce_prompt = hub.pull("rlm/reduce-prompt")

In [9]:
reduce_prompt

ChatPromptTemplate(input_variables=['doc_summaries'], metadata={'lc_hub_owner': 'rlm', 'lc_hub_repo': 'reduce-prompt', 'lc_hub_commit_hash': 'a3d558b35e478278c448c2988cd2ed1422cede59d59c63cf203b733d4ddf73f0'}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['doc_summaries'], template='The following is set of summaries:\n{doc_summaries}\nTake these and distill it into a final, consolidated summary of the main themes. \nHelpful Answer:'))])

In [25]:
# Run chain
reduce_chain = LLMChain(llm=llm, prompt=reduce_prompt)

# Takes a list of documents, combines them into a single string, and passes this to an LLMChain
combine_documents_chain = StuffDocumentsChain(
    llm_chain=reduce_chain, document_variable_name="doc_summaries"  # added/edited
)

# Combines and iteratively reduces the mapped documents
reduce_documents_chain = ReduceDocumentsChain(
    # This is final chain that is called.
    combine_documents_chain=combine_documents_chain,
    # If documents exceed context for `StuffDocumentsChain`
    collapse_documents_chain=combine_documents_chain,
    # The maximum number of tokens to group documents into.
    token_max=4000,
)

In [26]:
# Combining documents by mapping a chain over them, then combining results
map_reduce_chain = MapReduceDocumentsChain(
    # Map chain
    llm_chain=map_chain,
    # Reduce chain
    reduce_documents_chain=reduce_documents_chain,
    # The variable name in the llm_chain to put the documents in
    document_variable_name="docs",
    # Return the results of the map steps in the output
    return_intermediate_steps=False,
)

text_splitter = CharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=1000, chunk_overlap=0
)
split_docs = text_splitter.split_documents(docs)

Created a chunk of size 1003, which is longer than the specified 1000


In [27]:
result = map_reduce_chain.invoke(split_docs)

print(result["output_text"])

 Overall, the main themes revolve around the potential and challenges of incorporating LLMs into autonomous agent systems for various applications.


### Go deeper

## Option 3. Refine

In [28]:
chain = load_summarize_chain(llm, chain_type="refine")
result = chain.invoke(split_docs)

print(result["output_text"])



The GPT-Engineer project aims to create a repository of code given a task specified in natural language, using LLM-powered agents. These agents have the potential to revolutionize knowledge-intensive and decision-making tasks through continuous performance evaluation, self-criticism, and refinement strategies. However, challenges and limitations still exist, including restricted context capacity, long-term planning, and reliability of natural language interfaces. Further research and refinement are needed to fully harness the potential of LLM-powered agents, as demonstrated by recent advancements in adversarial attacks, prompting techniques, and prompt engineering.


In [29]:
prompt_template = """Write a concise summary of the following:
{text}
CONCISE SUMMARY:"""
prompt = PromptTemplate.from_template(prompt_template)

refine_template = (
    "Your job is to produce a final summary\n"
    "We have provided an existing summary up to a certain point: {existing_answer}\n"
    "We have the opportunity to refine the existing summary"
    "(only if needed) with some more context below.\n"
    "------------\n"
    "{text}\n"
    "------------\n"
    "Given the new context, refine the original summary in Italian"
    "If the context isn't useful, return the original summary."
)
refine_prompt = PromptTemplate.from_template(refine_template)
chain = load_summarize_chain(
    llm=llm,
    chain_type="refine",
    question_prompt=prompt,
    refine_prompt=refine_prompt,
    return_intermediate_steps=True,
    input_key="input_documents",
    output_key="output_text",
)
result = chain.invoke({"input_documents": split_docs}, return_only_outputs=True)

In [30]:
print(result["output_text"])



Il concetto di LLM Powered Autonomous Agents si basa sull'utilizzo dei grandi modelli linguistici (LLM) come controller principale per la costruzione di agenti intelligenti. Questi agenti, guidati da LLM, sono in grado di suddividere compiti complessi in obiettivi più piccoli, riflettere su se stessi e imparare dalle azioni passate, utilizzando sia la memoria a breve che a lungo termine. Hanno anche la capacità di accedere ad API esterne per informazioni aggiuntive e di interagire con altri agenti. L'approccio è stato dimostrato attraverso vari esempi di proof-of-concept, come il sistema AutoGPT, e ha il potenziale per essere un potente risolutore di problemi generali. Tuttavia, sono presenti sfide come la limitata capacità di comunicazione e il potenziale di errori nella comprensione del linguaggio naturale da parte dei LLM. Inoltre, la pianificazione a lungo termine e la decomposizione dei compiti rimangono ancora delle sf


In [31]:
print("\n\n".join(result["intermediate_steps"][:3]))


LLM Powered Autonomous Agents is a concept where large language models (LLM) are used as the core controller for building intelligent agents. These agents are capable of breaking down complex tasks into smaller subgoals, self-reflecting and learning from past actions, and utilizing both short-term and long-term memory. They also have the ability to access external APIs for additional information. This approach has been demonstrated through various proof-of-concept examples and has the potential to be a powerful general problem solver.


LLM Powered Autonomous Agents è un concetto in cui i grandi modelli linguistici (LLM) sono utilizzati come controller principale per la costruzione di agenti intelligenti. Questi agenti sono in grado di suddividere compiti complessi in obiettivi più piccoli, riflettere su se stessi e imparare dalle azioni passate, utilizzando sia la memoria a breve che a lungo termine. Hanno anche la capacità di accedere ad API esterne per informazioni aggiuntive. Ques

## Splitting and summarizing in a single chain

In [32]:
from langchain.chains import AnalyzeDocumentChain

summarize_document_chain = AnalyzeDocumentChain(
    combine_docs_chain=chain, text_splitter=text_splitter
)
summarize_document_chain.invoke(docs[0].page_content)

Created a chunk of size 1003, which is longer than the specified 1000
